## **Análisis componentes para crear el nodo maestro `hadoop-master`**

#### **`Dockerfile`**

**`FROM ubuntu:latest`**: Esta línea indica que la imagen que se va a construir se basará en la imagen base de Ubuntu, específicamente la versión "latest" (última). En Docker, una imagen base proporciona el sistema de archivos inicial y los binarios necesarios para ejecutar una aplicación. En este caso, la imagen base es Ubuntu, lo que significa que la imagen de Docker contendrá un sistema de archivos similar al de Ubuntu.

**`MAINTAINER Alfonso Perez Lino`**: Esta línea se utiliza para especificar el nombre del mantenedor o creador de la imagen. En versiones anteriores de Docker, esta línea era importante para proporcionar información sobre quién mantenía o creaba la imagen. Sin embargo, a partir de Docker 1.13, la instrucción **MAINTAINER** está obsoleta en favor de las etiquetas **LABEL**. Ahora, la información del mantenedor se suele agregar como una etiqueta de metadatos en lugar de utilizar la instrucción **MAINTAINER**.

**`LABEL creador="Alfonso Perez Lino"`**: Esta línea agrega metadatos a la imagen resultante. Los metadatos proporcionan información adicional sobre la imagen, como el nombre del creador, la versión, la descripción, etc. En este caso, se está utilizando la etiqueta creador para indicar el nombre del creador de la imagen. Esta información puede ser útil para otros usuarios que trabajen con la imagen, ya que proporciona contexto sobre su origen y su propósito.

In [ ]:
FROM hadoop-cluster-base
LABEL Alfonso Perez Lino

**`WORKDIR /root`**: Este comando establece el directorio de trabajo actual dentro del contenedor. **WORKDIR** se utiliza para cambiar el directorio de trabajo en el contenedor para el resto de los comandos en el Dockerfile. En este caso, se establece el directorio de trabajo como **/root**. Esto significa que todos los comandos posteriores en el Dockerfile se ejecutarán en el contexto de este directorio dentro del contenedor. Esto puede ser útil para organizar el espacio de trabajo y las operaciones del contenedor.

In [ ]:
WORKDIR /root

Estas líneas en un Dockerfile están relacionadas con la construcción de una imagen de Docker. Veamos qué hacen cada una de ellas:

- `ADD config/bootstrap.sh /etc/bootstrap.sh`: Esta línea copia el archivo bootstrap.sh desde el directorio config de tu contexto de construcción de Docker al directorio /etc dentro de la imagen que estás construyendo. En resumen, toma el archivo bootstrap.sh y lo coloca en /etc/bootstrap.sh dentro de la imagen.

- `RUN chown root:root /etc/bootstrap.sh`: Esta instrucción RUN ejecuta un comando dentro del contenedor que se está construyendo. En este caso, chown root:root /etc/bootstrap.sh establece el propietario y el grupo del archivo /etc/bootstrap.sh como root. Esto es una práctica común para archivos importantes del sistema, ya que asegura que solo el usuario root pueda modificarlos.

- `RUN chmod 700 /etc/bootstrap.sh`: Esta instrucción RUN también ejecuta un comando dentro del contenedor en construcción. chmod 700 /etc/bootstrap.sh establece los permisos del archivo /etc/bootstrap.sh para que solo el propietario (root, en este caso) tenga permisos de lectura, escritura y ejecución, mientras que ningún otro usuario tenga permisos.

En resumen, estas líneas en el Dockerfile están configurando un archivo bootstrap.sh en la imagen Docker, asegurándose de que solo el usuario root tenga acceso y control sobre él, lo que es una práctica de seguridad común. Este archivo bootstrap.sh probablemente desempeñe un papel importante en la inicialización o configuración del contenedor cuando se ejecute.

In [ ]:
ADD config/bootstrap.sh /etc/bootstrap.sh
RUN chown root:root /etc/bootstrap.sh
RUN chmod 700 /etc/bootstrap.sh

Esta línea en un Dockerfile establece una variable de entorno llamada BOOTSTRAP con el valor /etc/bootstrap.sh. Veamos qué significa esto:

- `ENV BOOTSTRAP /etc/bootstrap.sh`: Esta instrucción ENV se utiliza para definir una variable de entorno en la imagen Docker que estás construyendo. En este caso, se define la variable BOOTSTRAP con el valor /etc/bootstrap.sh. Esto significa que cuando se ejecute un contenedor basado en esta imagen, la variable de entorno BOOTSTRAP estará disponible dentro del contenedor y tendrá el valor /etc/bootstrap.sh. Esta variable de entorno puede ser utilizada por aplicaciones o scripts dentro del contenedor para referenciar el archivo bootstrap.sh de manera más conveniente, ya que no necesitan conocer la ruta completa del archivo, solo necesitan leer el valor de la variable BOOTSTRAP.

Por ejemplo, si tienes un script dentro del contenedor que necesita ejecutar /etc/bootstrap.sh, en lugar de escribir la ruta completa, simplemente puede hacer referencia a la variable de entorno $BOOTSTRAP, lo que hace que el script sea más portátil y adaptable a diferentes entornos.

In [ ]:
ENV BOOTSTRAP /etc/bootstrap.sh

Esta línea en un Dockerfile establece el comando predeterminado que se ejecutará cuando inicies un contenedor basado en la imagen que estás construyendo. Veamos qué significa esta instrucción:

- `CMD ["/etc/bootstrap.sh", "-d"]`: Esta instrucción `CMD` define el comando predeterminado que se ejecutará cuando inicies un contenedor basado en la imagen Docker que estás construyendo. En este caso, el comando predeterminado es `/etc/bootstrap.sh -d`.

Esto significa que cuando inicies un contenedor basado en esta imagen, Docker ejecutará automáticamente el script `/etc/bootstrap.sh` con el argumento `-d`. La opción `-d` podría tener un significado específico para el script `bootstrap.sh`, como iniciar el script en modo demonio.

En resumen, esta línea indica a Docker que cuando se inicie un contenedor basado en esta imagen, se debe ejecutar el script `/etc/bootstrap.sh` con el argumento `-d` como el comando predeterminado del contenedor.

In [ ]:
CMD ["/etc/bootstrap.sh", "-d"]

### **Config-files/`bootstrap.sh`**

Este parece ser un script de inicio para un entorno de Hadoop en un sistema Linux. Veamos línea por línea:

```bash
#!/bin/bash
```
Esta línea es un shebang. Indica que el script debe ser interpretado por Bash, el intérprete de comandos de Unix.

```bash
rm /tmp/*.pid
```
Este comando elimina cualquier archivo con extensión `.pid` en el directorio `/tmp`. Los archivos `.pid` suelen ser archivos de identificador de proceso que pueden dejar los servicios cuando se cierran incorrectamente. Son archivos de identificación de proceso (PID). Estos archivos se utilizan para almacenar el ID de proceso (PID) del proceso principal del servicio del nodo de Hadoop. En Hadoop, durante el proceso de inicio del nodo, se genera un PID que identifica su proceso principal en el sistema operativo. Este PID se almacena en el archivo ".pid" en el directorio "/tmp". La función principal de este archivo es permitir que otros scripts o herramientas puedan identificar y manipular el proceso del nodo. Por ejemplo, si necesitas detener o reiniciar el servicio del nodo, podrías usar este PID para enviar la señal adecuada al proceso correspondiente. Es importante tener en cuenta que este archivo se crea y actualiza automáticamente por Hadoop durante el inicio del servicio del nodo y se elimina cuando el servicio se detiene. Si el servicio del nodo se detiene inesperadamente o si el archivo PID queda huérfano, puede ser necesario eliminar manualmente este archivo para evitar conflictos durante el próximo inicio del servicio.

```bash
service ssh start
```
Este comando inicia el servicio SSH. SSH es probablemente necesario para la administración remota del sistema o para la comunicación entre los nodos de un clúster Hadoop.

```bash
$HADOOP_HOME/sbin/start-dfs.sh
```
Este comando ejecuta el script `start-dfs.sh` ubicado en el directorio de inicio de Hadoop (`$HADOOP_HOME`). Este script se utiliza para iniciar el sistema de archivos distribuido de Hadoop (HDFS).

```bash
$HADOOP_HOME/sbin/start-yarn.sh 
```
Este comando ejecuta el script `start-yarn.sh` ubicado en el directorio de inicio de Hadoop (`$HADOOP_HOME`). Este script se utiliza para iniciar YARN, que es el administrador de recursos de Hadoop.

```bash
bash
```
Esta línea simplemente invoca un nuevo shell de Bash. Después de ejecutar todos los comandos anteriores, el script se detendría y dejaría al usuario en un shell interactivo de Bash, probablemente para realizar más acciones o para mantener el contenedor en ejecución.

En resumen, este script se encarga de realizar algunas configuraciones iniciales necesarias para un entorno de Hadoop, como iniciar los servicios de SSH, HDFS y YARN, y luego dejar al usuario en un shell interactivo.

In [ ]:
#!/bin/bash
rm /tmp/*.pid


service ssh start
$HADOOP_HOME/sbin/start-dfs.sh
$HADOOP_HOME/sbin/start-yarn.sh 

bash